___

## Day 2: Document Chunking

Now that we have documents from Day 1, we need to split them into smaller chunks for the RAG pipeline. Large documents don't fit in LLM context windows, so we chunk them with overlap to preserve context.

We'll start with **sliding window chunking** - the simplest and often most effective approach. We'll also add token counting to measure chunk sizes accurately.

In [9]:
# Day 2 imports
import copy
import tiktoken
import re

### Token Counting

LLMs charge by **tokens**, not characters. A token is roughly 4 characters on average, but varies by content. We use `tiktoken` (OpenAI's official tokenizer) to count tokens accurately.

The `cl100k_base` encoding is used by GPT-4, GPT-3.5-turbo, and text-embedding-ada-002 - the models we'll use for RAG.

In [10]:
def count_tokens(text, encoding='cl100k_base'):
    """Count tokens in text using tiktoken.

    Args:
        text: Text to count tokens for
        encoding: Tiktoken encoding (default cl100k_base for GPT-4/3.5)

    Returns:
        Number of tokens
    """
    enc = tiktoken.get_encoding(encoding)
    return len(enc.encode(text))

# Test it
sample = "Hello, world! This is a test of token counting."
print(f"Sample text: '{sample}'")
print(f"Characters: {len(sample)}")
print(f"Tokens: {count_tokens(sample)}")

Sample text: 'Hello, world! This is a test of token counting.'
Characters: 47
Tokens: 12


### Sliding Window Chunking

Sliding window splits text into fixed-size chunks with overlap:
- **chunk_size**: How many characters per chunk (default 2000 ≈ 512 tokens)
- **overlap**: How many characters overlap between chunks (default 1000)

Overlap ensures context isn't lost at chunk boundaries. If a concept spans the boundary, it appears in both chunks.

In [11]:
def chunk_sliding_window(doc, chunk_size=2000, overlap=1000):
    """Split document into overlapping chunks using sliding window.

    Args:
        doc: Document dict with 'content', 'filename', 'metadata' keys
        chunk_size: Characters per chunk (default 2000 ≈ 512 tokens)
        overlap: Characters overlap between chunks (default 1000)

    Returns:
        List of chunk dicts with preserved metadata

    Raises:
        ValueError: If required keys missing or invalid parameters
    """
    # Validation (D-11)
    if 'content' not in doc or 'filename' not in doc:
        raise ValueError("Document must have 'content' and 'filename' keys")
    if chunk_size <= overlap:
        raise ValueError("chunk_size must be greater than overlap")
    if overlap < 0:
        raise ValueError("overlap must be non-negative")

    content = doc['content']
    chunks = []
    start = 0

    while start < len(content):
        end = start + chunk_size
        chunk_content = content[start:end]

        # Create chunk with metadata preservation (D-12, D-13, D-14)
        chunk = {
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': chunk_content,
            'chunk_id': f"{doc['filename']}-chunk-{len(chunks)}",
            'chunk_index': len(chunks),
            'total_chunks': -1,  # Updated after loop
            'chunk_method': 'sliding_window'
        }
        chunks.append(chunk)
        start += (chunk_size - overlap)

    # Update total_chunks for all
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

### Testing Chunking on Day 1 Data

Let's apply sliding window chunking to documents we loaded in Day 1. We'll look at chunk counts, sizes, and verify metadata is preserved.

In [12]:
# Chunk all documents from Day 1
sliding_window_chunks = []
for doc in evidently_docs:
    chunks = chunk_sliding_window(doc, chunk_size=2000, overlap=1000)
    sliding_window_chunks.extend(chunks)

print(f"Total documents: {len(evidently_docs)}")
print(f"Total chunks: {len(sliding_window_chunks)}")
print(f"Average chunks per doc: {len(sliding_window_chunks) / len(evidently_docs):.1f}")

Total documents: 95
Total chunks: 648
Average chunks per doc: 6.8


In [13]:
# Show first 3 chunks with character and token counts
for chunk in sliding_window_chunks[:3]:
    char_count = len(chunk['content'])
    token_count = count_tokens(chunk['content'])
    print(f"Chunk: {chunk['chunk_id']}")
    print(f"  Size: {char_count:,} chars, {token_count} tokens")
    print(f"  Metadata preserved: {bool(chunk['metadata'])}")
    print(f"  Position: {chunk['chunk_index'] + 1} of {chunk['total_chunks']}")
    print(f"  Preview: {chunk['content'][:100]}...")
    print()

Chunk: docs-main/api-reference/introduction.mdx-chunk-0
  Size: 783 chars, 180 tokens
  Metadata preserved: True
  Position: 1 of 1
  Preview: <Note>
  If you're not looking to build API reference documentation, you can delete
  this section b...

Chunk: docs-main/changelog/changelog.mdx-chunk-0
  Size: 2,000 chars, 591 tokens
  Metadata preserved: True
  Position: 1 of 6
  Preview: <Update label="2025-07-18" description="Evidently v0.7.11">
  ## **Evidently 0.7.11**

  Full releas...

Chunk: docs-main/changelog/changelog.mdx-chunk-1
  Size: 2,000 chars, 665 tokens
  Metadata preserved: True
  Position: 2 of 6
  Preview: ntly/blob/main/examples/cookbook/prompt_optimization_bookings_example.ipynb)
- Tweet generation prom...



### Observations

- **Token/character ratio**: Roughly 1 token per 4 characters, but varies by content
- **Metadata preservation**: All frontmatter from Day 1 is preserved in each chunk
- **Chunk IDs**: Human-readable, sortable, traceable back to source document
- **Overlap effect**: Adjacent chunks share 1000 characters, preserving context at boundaries

**Limitation**: Sliding window is structure-blind - it will break code blocks and tables mid-content. We'll address this with structure-aware chunking strategies in later phases.

### Paragraph-Based Chunking

Instead of fixed-size windows, paragraph chunking respects natural language boundaries. We split on blank lines (`\n\s*\n` pattern), keeping each paragraph as an atomic unit.

**Tradeoffs:**
- Preserves semantic completeness within each chunk
- Chunks vary widely in size (a paragraph can be 1 sentence or 20)
- May produce very small chunks (single-line paragraphs)
- No overlap between chunks (each paragraph is independent)

In [14]:
def chunk_by_paragraph(doc):
    """Split document into paragraph chunks.

    Splits on blank lines (\\n\\s*\\n pattern), filters empty paragraphs,
    and preserves all metadata from the source document.
    """
    # Split on \n\s*\n pattern - matches paragraph boundaries
    paragraphs = re.split(r'\n\s*\n', doc['content'])

    # Filter empty and trim whitespace (D-02, D-04)
    paragraphs = [p.strip() for p in paragraphs if p.strip()]

    # Create chunks with metadata preservation (D-05)
    chunks = []
    for i, para_text in enumerate(paragraphs):
        chunk = {
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': para_text,
            'chunk_id': f"{doc['filename']}-chunk-{i}",
            'chunk_index': i,
            'total_chunks': len(paragraphs),
            'chunk_method': 'paragraph'
        }
        chunks.append(chunk)

    return chunks

### Section-Based Chunking

Section chunking uses markdown structure to find logical boundaries. We split on level 2 headers (`## Header`), keeping each section with its nested subsections.

**Tradeoffs:**
- Preserves topic coherence (each section = one concept)
- Header text included in chunk (provides context for retrieval)
- Nested subsections (###, ####) stay with their parent section
- Chunks can be very large if sections are long
- Documents without ## headers return as single chunk

In [15]:
def chunk_by_section(doc):
    """Split document by ## markdown headers.

    Each chunk contains one ## section with all nested subsections
    until the next ## header. Header text is included in the chunk.
    """
    # Find all ## headers and their positions (D-07, D-08)
    pattern = r'^##\s+(.+)$'
    matches = list(re.finditer(pattern, doc['content'], re.MULTILINE))

    if not matches:
        # No sections found, return whole doc as single chunk
        return [{
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': doc['content'],
            'chunk_id': f"{doc['filename']}-chunk-0",
            'chunk_index': 0,
            'total_chunks': 1,
            'chunk_method': 'section'
        }]

    chunks = []
    for i, match in enumerate(matches):
        start = match.start()
        # End is start of next section or end of document (D-09)
        end = matches[i+1].start() if i+1 < len(matches) else len(doc['content'])

        section_text = doc['content'][start:end].strip()

        # Skip empty sections (D-10)
        if not section_text:
            continue

        chunk = {
            'filename': doc['filename'],
            'metadata': copy.deepcopy(doc.get('metadata', {})),
            'content': section_text,  # includes header (D-06)
            'chunk_id': f"{doc['filename']}-chunk-{len(chunks)}",
            'chunk_index': len(chunks),
            'total_chunks': -1,  # Updated after loop
            'chunk_method': 'section'
        }
        chunks.append(chunk)

    # Update total_chunks for all (D-12)
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

### Testing Paragraph and Section Chunking

Let's apply both strategies to the Evidently docs and compare chunk counts.

In [16]:
# Apply paragraph chunking to all docs
paragraph_chunks = []
for doc in evidently_docs:
    paragraph_chunks.extend(chunk_by_paragraph(doc))

print(f"Paragraph chunking: {len(evidently_docs)} docs -> {len(paragraph_chunks)} chunks")
print(f"Average: {len(paragraph_chunks) / len(evidently_docs):.1f} chunks per doc")

# Apply section chunking to all docs
section_chunks = []
for doc in evidently_docs:
    section_chunks.extend(chunk_by_section(doc))

print(f"\nSection chunking: {len(evidently_docs)} docs -> {len(section_chunks)} chunks")
print(f"Average: {len(section_chunks) / len(evidently_docs):.1f} chunks per doc")

Paragraph chunking: 95 docs -> 3314 chunks
Average: 34.9 chunks per doc

Section chunking: 95 docs -> 293 chunks
Average: 3.1 chunks per doc


### Sample Chunks

Let's examine a sample chunk from each strategy to verify metadata preservation.

In [17]:
# Sample paragraph chunk
if paragraph_chunks:
    sample = paragraph_chunks[0]
    print("=== Paragraph Chunk Sample ===")
    print(f"chunk_id: {sample['chunk_id']}")
    print(f"chunk_index: {sample['chunk_index']} of {sample['total_chunks']}")
    print(f"chunk_method: {sample['chunk_method']}")
    print(f"filename: {sample['filename']}")
    print(f"metadata keys: {list(sample['metadata'].keys())}")
    print(f"content preview: {sample['content'][:200]}...")
    print()

# Sample section chunk
if section_chunks:
    sample = section_chunks[0]
    print("=== Section Chunk Sample ===")
    print(f"chunk_id: {sample['chunk_id']}")
    print(f"chunk_index: {sample['chunk_index']} of {sample['total_chunks']}")
    print(f"chunk_method: {sample['chunk_method']}")
    print(f"filename: {sample['filename']}")
    print(f"metadata keys: {list(sample['metadata'].keys())}")
    print(f"content preview: {sample['content'][:200]}...")

=== Paragraph Chunk Sample ===
chunk_id: docs-main/api-reference/introduction.mdx-chunk-0
chunk_index: 0 of 7
chunk_method: paragraph
filename: docs-main/api-reference/introduction.mdx
metadata keys: ['title', 'description']
content preview: <Note>
  If you're not looking to build API reference documentation, you can delete
  this section by removing the api-reference folder.
</Note>...

=== Section Chunk Sample ===
chunk_id: docs-main/api-reference/endpoint/create.mdx-chunk-0
chunk_index: 0 of 1
chunk_method: section
filename: docs-main/api-reference/endpoint/create.mdx
metadata keys: ['title', 'openapi']
content preview: ...


### Verifying Nested Subsection Preservation

Per D-09, section chunks must include all nested subsections (###, ####) within each ## section. Let's verify this by checking for ### headers inside section chunks.

In [18]:
# Verify nested subsections are preserved in section chunks (D-09)
nested_header_pattern = r'^###\s+.+$'

# Find section chunks that contain nested ### headers
chunks_with_nested = []
for chunk in section_chunks:
    nested_matches = re.findall(nested_header_pattern, chunk['content'], re.MULTILINE)
    if nested_matches:
        chunks_with_nested.append({
            'chunk_id': chunk['chunk_id'],
            'nested_headers': nested_matches[:3],  # Show first 3
            'nested_count': len(nested_matches)
        })

print(f"Section chunks containing nested ### headers: {len(chunks_with_nested)} of {len(section_chunks)}")
print()

# Show examples of preserved nested structure
if chunks_with_nested:
    print("Examples of nested subsection preservation:")
    for item in chunks_with_nested[:3]:
        print(f"\n  {item['chunk_id']}")
        print(f"    Nested headers ({item['nested_count']} total): {item['nested_headers']}")

# Assertion: If any doc has nested headers, at least some section chunks should preserve them
# (This confirms D-09 behavior - nested subsections stay with parent ## section)
docs_with_nested = sum(1 for doc in evidently_docs if re.search(r'^###\s+', doc['content'], re.MULTILINE))
if docs_with_nested > 0:
    assert len(chunks_with_nested) > 0, "D-09 violation: nested ### headers not found in any section chunk"
    print(f"\nD-09 VERIFIED: Nested subsections preserved in section chunks")
else:
    print("\nNote: No nested ### headers found in source docs (D-09 not applicable)")

Section chunks containing nested ### headers: 52 of 293

Examples of nested subsection preservation:

  docs-main/docs/library/data_definition.mdx-chunk-0
    Nested headers (1 total): ['### Special cases']

  docs-main/docs/library/data_definition.mdx-chunk-1
    Nested headers (6 total): ['### Column types', '### ID and timestamp', '### LLM evals']

  docs-main/docs/library/leftover_content.mdx-chunk-0
    Nested headers (7 total): ['### Diversity', '### Novelty', '### Serendipity']

D-09 VERIFIED: Nested subsections preserved in section chunks


### Strategy Comparison Framework

Now let's compare all three chunking strategies side-by-side using concrete metrics:
- **Basic counts:** Total chunks, average per document
- **Token statistics:** Average, min, max tokens per chunk
- **Size distribution:** 25th, 50th, 75th, 95th percentiles

In [19]:
import statistics
import tiktoken

def compare_chunking_strategies(strategies):
    """Compare chunking strategies across multiple metrics.

    Args:
        strategies: Dict mapping strategy name to list of chunks
                   e.g., {'sliding_window': chunks1, 'paragraph': chunks2}

    Returns:
        Formatted markdown table string with strategies as columns, metrics as rows
    """
    enc = tiktoken.get_encoding('cl100k_base')

    results = {}
    for name, chunks in strategies.items():
        if not chunks:
            results[name] = {metric: 0 for metric in [
                'total_chunks', 'avg_per_doc', 'avg_tokens', 'min_tokens',
                'max_tokens', 'token_char_ratio', 'p25_chars', 'p50_chars',
                'p75_chars', 'p95_chars'
            ]}
            continue

        char_counts = [len(c['content']) for c in chunks]
        token_counts = [len(enc.encode(c['content'])) for c in chunks]

        # Count documents (unique filenames)
        doc_count = len(set(c['filename'] for c in chunks))

        # Calculate percentiles with edge case handling
        if len(char_counts) > 1:
            quartiles = statistics.quantiles(char_counts, n=4)
            p25 = quartiles[0]
            p75 = quartiles[2]
            # p95 requires n=20 ventiles
            ventiles = statistics.quantiles(char_counts, n=20)
            p95 = ventiles[18] if len(ventiles) > 18 else char_counts[-1]
        else:
            p25 = p75 = p95 = char_counts[0] if char_counts else 0

        p50 = statistics.median(char_counts) if char_counts else 0

        results[name] = {
            'total_chunks': len(chunks),
            'avg_per_doc': len(chunks) / doc_count if doc_count > 0 else 0,
            'avg_tokens': statistics.mean(token_counts) if token_counts else 0,
            'min_tokens': min(token_counts) if token_counts else 0,
            'max_tokens': max(token_counts) if token_counts else 0,
            'token_char_ratio': statistics.mean(token_counts) / statistics.mean(char_counts) if char_counts else 0,
            'p25_chars': p25,
            'p50_chars': p50,
            'p75_chars': p75,
            'p95_chars': p95,
        }

    # Format as markdown table
    strategy_names = list(strategies.keys())
    metrics = ['total_chunks', 'avg_per_doc', 'avg_tokens', 'min_tokens',
               'max_tokens', 'token_char_ratio', 'p25_chars', 'p50_chars',
               'p75_chars', 'p95_chars']

    # Header row
    header = "| Metric | " + " | ".join(strategy_names) + " |"
    separator = "|--------|" + "|".join(["--------"] * len(strategy_names)) + "|"

    # Data rows
    rows = []
    for metric in metrics:
        row_vals = []
        for name in strategy_names:
            val = results[name][metric]
            if isinstance(val, float):
                row_vals.append(f"{val:.2f}")
            else:
                row_vals.append(str(val))
        rows.append(f"| {metric} | " + " | ".join(row_vals) + " |")

    table = "\n".join([header, separator] + rows)
    return table

In [20]:
# Collect all chunks from Plan 01
# sliding_window_chunks already exists from Phase 7

strategies = {
    'sliding_window': sliding_window_chunks,
    'paragraph': paragraph_chunks,
    'section': section_chunks
}

comparison_table = compare_chunking_strategies(strategies)
print(comparison_table)

| Metric | sliding_window | paragraph | section |
|--------|--------|--------|--------|
| total_chunks | 648 | 3314 | 293 |
| avg_per_doc | 7.12 | 36.42 | 3.08 |
| avg_tokens | 366.12 | 38.02 | 411.37 |
| min_tokens | 2 | 1 | 0 |
| max_tokens | 819 | 1654 | 5353 |
| token_char_ratio | 0.21 | 0.21 | 0.21 |
| p25_chars | 1990.75 | 41.00 | 384.00 |
| p50_chars | 2000.00 | 87.00 | 851 |
| p75_chars | 2000.00 | 165.00 | 2100.50 |
| p95_chars | 2000.00 | 454.25 | 8259.90 |


### Comparison Observations

Key observations from the comparison:
- **Chunk count variance:** Paragraph chunking typically produces the most chunks (many small paragraphs), section chunking the fewest (large logical sections), sliding window in between
- **Token distribution:** Sliding window has most consistent token counts (by design), semantic strategies have high variance
- **Size outliers:** Section chunks can be very large if a section spans many pages
- **When to use each:**
  - Sliding window: When consistent chunk sizes matter (embedding models with fixed context)
  - Paragraph: When natural text boundaries are important (prose, narratives)
  - Section: When topic coherence matters (structured documentation)

### Chunk Quality Inspection

The comparison framework tells us *how many* chunks we get, but not *how good* they are. Let's add a quality inspector that flags potential problems:
- **Boundary problems:** Mid-word breaks, unclosed code blocks
- **Size outliers:** Chunks much larger or smaller than average
- **Content completeness:** Orphaned references ("as shown above", "see below")

In [21]:
import re
import statistics

def inspect_chunk_quality(chunks, strategy_name):
    """Detect quality issues in chunks.

    Args:
        chunks: List of chunk dicts
        strategy_name: Name of chunking strategy (for context)

    Returns:
        List of flagged samples with issue descriptions.
        Limited to 3-5 samples for readability.
    """
    if not chunks:
        return []

    flagged = []
    chunk_sizes = [len(c['content']) for c in chunks]
    mean_size = statistics.mean(chunk_sizes)

    for chunk in chunks:
        issues = []
        text = chunk['content']

        # Boundary problems: incomplete sentence at end
        # Check if ends with sentence-ending punctuation or markdown structure
        if text and not text.rstrip().endswith(('.', '!', '?', '```', ')', ']', ':', '-')):
            if not text.strip().startswith(('##', '-', '*', '1.', '>')):
                issues.append('mid-word or incomplete sentence')

        # Broken code blocks: odd number of ``` markers
        if text.count('```') % 2 != 0:
            issues.append('unclosed code block')

        # Orphaned references: words that suggest missing context
        if re.search(r'\b(above|below|previous|next|following)\b', text.lower()):
            issues.append('orphaned reference')

        # Size outliers: >2x or <0.5x mean
        size = len(text)
        if size > mean_size * 2:
            issues.append(f'too large ({size / mean_size:.1f}x mean)')
        elif size < mean_size * 0.5:
            issues.append(f'too small ({size / mean_size:.2f}x mean)')

        if issues:
            flagged.append({'chunk': chunk, 'issues': issues})

    # Return first 5 samples (D-17)
    return flagged[:5]

### Quality Inspection Results

Let's inspect each strategy for quality issues.

In [22]:
for name, chunks in strategies.items():
    print(f"\n=== {name.upper()} Quality Inspection ===")
    flagged = inspect_chunk_quality(chunks, name)

    if not flagged:
        print("No quality issues detected in sample.")
    else:
        print(f"Found {len(flagged)} potentially problematic chunks:")
        for i, item in enumerate(flagged):
            chunk = item['chunk']
            issues = item['issues']
            print(f"\n  [{i+1}] {chunk['chunk_id']}")
            print(f"      Issues: {', '.join(issues)}")
            print(f"      Preview: {chunk['content'][:100]}...")


=== SLIDING_WINDOW Quality Inspection ===
Found 5 potentially problematic chunks:

  [1] docs-main/api-reference/introduction.mdx-chunk-0
      Issues: orphaned reference, too small (0.45x mean)
      Preview: <Note>
  If you're not looking to build API reference documentation, you can delete
  this section b...

  [2] docs-main/changelog/changelog.mdx-chunk-0
      Issues: mid-word or incomplete sentence
      Preview: <Update label="2025-07-18" description="Evidently v0.7.11">
  ## **Evidently 0.7.11**

  Full releas...

  [3] docs-main/changelog/changelog.mdx-chunk-1
      Issues: mid-word or incomplete sentence
      Preview: ntly/blob/main/examples/cookbook/prompt_optimization_bookings_example.ipynb)
- Tweet generation prom...

  [4] docs-main/changelog/changelog.mdx-chunk-2
      Issues: mid-word or incomplete sentence
      Preview: eleases/tag/v0.7.6).
</Update>

<Update label="2025-05-09" description="Evidently v0.7.5">
  ## **Ev...

  [5] docs-main/changelog/changelog.mdx-ch

### Quality Inspection Observations

**Important:** These heuristics flag *potential* issues, not definite problems:
- "orphaned reference" can be a false positive if the context is self-contained
- "mid-word or incomplete sentence" may flag code snippets or bullet points
- "unclosed code block" is usually a real problem that breaks rendering

**Which strategy has fewer issues?**
- Semantic strategies (paragraph, section) typically have fewer boundary problems
- Sliding window may split mid-sentence or mid-code-block
- Section chunking may flag size outliers (very large sections)

Use these flags as starting points for manual review, not automatic rejection criteria.

### LLM-Based Chunking

Use LLM intelligence to identify semantic boundaries. Instead of fixed patterns (sliding window) or structure markers (paragraphs, sections), we ask an LLM to analyze the document and mark where natural topic boundaries occur.

**Key tradeoffs:**
- **Cost**: API calls cost money (Groq ~$0.05/1M input tokens, OpenAI ~$0.15/1M)
- **Quality**: LLM can understand context and preserve semantic units
- **Speed**: Much slower than regex-based methods

In [ ]:
# LLM chunking imports
import os
import time
import json
from openai import OpenAI
from groq import Groq

In [ ]:
def split_at_boundaries(doc, boundaries, provider):
    """Split document at given character positions.

    Args:
        doc: Source document dict
        boundaries: List of character positions for splits
        provider: 'groq' or 'openai' (for chunk_method label)

    Returns:
        List of chunk dicts with metadata preservation
    """
    content = doc['content']
    chunks = []

    # Add start (0) and end (len) to boundaries
    positions = [0] + sorted(boundaries) + [len(content)]

    for i in range(len(positions) - 1):
        start = positions[i]
        end = positions[i + 1]
        chunk_text = content[start:end].strip()

        if chunk_text:  # Skip empty
            chunk = {
                'filename': doc['filename'],
                'metadata': copy.deepcopy(doc.get('metadata', {})),
                'content': chunk_text,
                'chunk_id': f"{doc['filename']}-chunk-{i}",
                'chunk_index': i,
                'total_chunks': -1,  # Updated after loop
                'chunk_method': f'llm_{provider}'
            }
            chunks.append(chunk)

    # Update total_chunks for all
    total = len(chunks)
    for chunk in chunks:
        chunk['total_chunks'] = total

    return chunks

In [ ]:
def chunk_by_llm(doc, provider='groq'):
    """Split document using LLM semantic boundary detection.

    Uses OpenAI or Groq API to identify natural topic boundaries in text.
    Returns character positions for splits, preserving original text exactly.

    Args:
        doc: Document dict with 'content', 'filename', 'metadata' keys
        provider: 'groq' (default, free tier) or 'openai'

    Returns:
        List of chunk dicts with preserved metadata

    Raises:
        ValueError: If API key missing
    """
    # Step 1: Check API keys (per D-02, D-03)
    key_name = 'GROQ_API_KEY' if provider == 'groq' else 'OPENAI_API_KEY'
    api_key = os.getenv(key_name)
    if not api_key:
        raise ValueError(f"{key_name} not found - set via: op run --env-file=...")

    # Step 2: Initialize client and select model (per D-04, D-05)
    if provider == 'groq':
        client = Groq(api_key=api_key)
        model = 'llama-3.1-8b-instant'
        input_rate = 0.05   # $/1M tokens
        output_rate = 0.08  # $/1M tokens
    else:
        client = OpenAI(api_key=api_key)
        model = 'gpt-4o-mini'
        input_rate = 0.15   # $/1M tokens
        output_rate = 0.60  # $/1M tokens

    # Step 3: Build prompt (per D-06 through D-10)
    system_msg = "You are a document chunking expert."
    user_msg = f"""Identify semantic boundaries in this document.

Rules:
- Split at topic changes and after complete thoughts
- Avoid mid-sentence breaks
- Keep code blocks intact
- Aim for chunks around 500-1000 tokens, but prioritize semantic completeness

Return character positions as JSON: {{"boundaries": [120, 450, 890]}}

Document:
{doc['content']}"""

    # Step 4: Call API with retry logic (per D-14, D-15, D-16)
    for attempt in range(3):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[
                    {"role": "system", "content": system_msg},
                    {"role": "user", "content": user_msg}
                ],
                response_format={"type": "json_object"},
                temperature=0.3
            )

            # Parse JSON boundaries (per D-07)
            result = json.loads(response.choices[0].message.content)
            boundaries = result.get('boundaries', [])

            # Extract usage for cost tracking (per D-11, D-12)
            usage = response.usage
            total_tokens = usage.total_tokens
            cost = (usage.prompt_tokens * input_rate + usage.completion_tokens * output_rate) / 1_000_000

            print(f"Document: {doc['filename'][:50]}... - Tokens: {total_tokens}, Cost: ${cost:.6f}")

            # Split document at boundaries
            chunks = split_at_boundaries(doc, boundaries, provider)
            return chunks

        except Exception as e:
            if '429' in str(e):  # Rate limit (per D-15)
                retry_after = getattr(e, 'retry_after', 2 ** attempt)
                print(f"Rate limit, waiting {retry_after}s...")
                time.sleep(retry_after)
            elif 'json' in str(e).lower():  # JSON parse error (per D-16)
                print(f"JSON parse error, retrying with stricter prompt...")
                user_msg += "\n\nONLY output valid JSON."
            else:
                print(f"Attempt {attempt + 1} failed: {e}")
                time.sleep(2 ** attempt)

    # Fallback to paragraph chunking after 3 failures (per D-16)
    print(f"LLM chunking failed, falling back to paragraph chunking")
    return chunk_by_paragraph(doc)

### Running LLM Chunking

**Setup:** Run notebook with API key via OnePassword injection:
```bash
op run --env-file=.env jupyter notebook
```

**Note:** We'll use Groq (free tier) by default. OpenAI available with `provider='openai'`.

In [ ]:
# Test on a single document first to verify setup
# Skip if no API key available
if os.getenv('GROQ_API_KEY') or os.getenv('OPENAI_API_KEY'):
    provider = 'groq' if os.getenv('GROQ_API_KEY') else 'openai'
    sample_doc = evidently_docs[0]
    sample_chunks = chunk_by_llm(sample_doc, provider=provider)
    print(f"\nSample document chunked into {len(sample_chunks)} chunks")
    print(f"Chunk method: {sample_chunks[0]['chunk_method'] if sample_chunks else 'N/A'}")
else:
    print("No API key found. Set GROQ_API_KEY or OPENAI_API_KEY to run LLM chunking.")
    print("Skipping LLM chunking tests.")

In [ ]:
# Chunk all documents with LLM (if API available)
# Note: This will make API calls - costs apply!
llm_chunks = []
if os.getenv('GROQ_API_KEY') or os.getenv('OPENAI_API_KEY'):
    provider = 'groq' if os.getenv('GROQ_API_KEY') else 'openai'
    print(f"Using {provider} provider")
    print(f"Processing {len(evidently_docs)} documents...\n")

    for doc in evidently_docs:
        chunks = chunk_by_llm(doc, provider=provider)
        llm_chunks.extend(chunks)

    print(f"\n{'='*60}")
    print(f"Total: {len(llm_chunks)} chunks from {len(evidently_docs)} documents")
else:
    print("Skipping LLM chunking - no API key available.")

In [ ]:
# Compare all chunking strategies (including LLM if available)
if llm_chunks:
    strategies_with_llm = {
        'sliding_window': sliding_window_chunks,
        'paragraph': paragraph_chunks,
        'section': section_chunks,
        'llm': llm_chunks
    }
    comparison_with_llm = compare_chunking_strategies(strategies_with_llm)
    print(comparison_with_llm)
else:
    print("LLM chunks not available - showing previous comparison:")
    print(comparison_table)

In [ ]:
# Inspect LLM chunk quality
if llm_chunks:
    print("=== LLM Chunking Quality Inspection ===")
    llm_flagged = inspect_chunk_quality(llm_chunks, 'llm')

    if not llm_flagged:
        print("No quality issues detected in sample.")
    else:
        print(f"Found {len(llm_flagged)} potentially problematic chunks:")
        for i, item in enumerate(llm_flagged):
            chunk = item['chunk']
            issues = item['issues']
            print(f"\n  [{i+1}] {chunk['chunk_id']}")
            print(f"      Issues: {', '.join(issues)}")
            print(f"      Preview: {chunk['content'][:100]}...")
else:
    print("LLM chunks not available - skipping quality inspection.")

### LLM Chunking Observations

**Cost vs Quality Tradeoff:**
- LLM chunking costs real money (~$0.001-0.01 per document depending on size)
- Quality may be better for prose/narrative content where semantic understanding matters
- For well-structured markdown with clear headers, section chunking may be just as good for free

**When to use LLM chunking:**
- High-value documents where boundary quality critically affects retrieval
- Content without clear structural markers (prose, narratives)
- When you can afford the API costs

**When simpler methods work:**
- Documentation with markdown structure (section chunking is free and fast)
- When consistent chunk sizes matter more than semantic boundaries (sliding window)
- Large corpora where LLM costs would be prohibitive